# Technical Demonstration of Data Poisoning

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 40
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
ENRON_CSV = DATA_DIR / "enron_spam.csv"
MURLs_CSV = DATA_DIR / "malicious_phish.csv"

POISONING_BUDGETS = [0.0, 0.01, 0.05, 0.10, 0.15, 0.20, 0.30, 0.35, 0.40, 0.45] 
TARGETED_BUDGETS = [0.0, 0.01, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
MAX_TFIDF_FEATURES = 5000
TEST_SIZE = 0.20

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)

In [2]:
import os
import zipfile
import pandas as pd
from pathlib import Path
import kaggle

ENRON_KAGGLE_DATASET  = "marcelwiechmann/enron-spam-data"     
ENRON_CSV_NAME  = "enron_spam_data.csv"   
  

DATA_DIR = Path("data")


def _download_kaggle_dataset(dataset_slug: str, dest_dir: Path) -> None:
    dest_dir.mkdir(parents=True, exist_ok=True)
    print(f"[Kaggle] Downloading '{dataset_slug}' → {dest_dir}")
    kaggle.api.authenticate()
    kaggle.api.dataset_download_files(dataset_slug, path=str(dest_dir), unzip=True, quiet=False)

def _ensure_csv(dataset_slug: str, csv_name: str, dest_dir: Path) -> Path:
    """Return the path to *csv_name*, downloading the dataset first if needed."""
    csv_path = dest_dir / csv_name
    if not csv_path.exists():
        _download_kaggle_dataset(dataset_slug, dest_dir)
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Expected '{csv_name}' inside dataset '{dataset_slug}', "
            f"but it was not found in {dest_dir}. "
            "Check the filename constant at the top of this file."
        )
    return csv_path


def _load_csv(csv_path: Path, dataset_name: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # ── Enron: combine Subject + Message into a single text field ────
    if "Subject" in df.columns and "Message" in df.columns:
        df["text"] = df["Subject"].fillna("") + " " + df["Message"].fillna("")
        df["label"] = df["Spam/Ham"]

    
    if "url" in df.columns:
        df = df.rename(columns={"url": "text", "type": "label"})

    required = {"text", "label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"{dataset_name} CSV is missing columns {missing}. "
            f"Actual columns: {df.columns.tolist()}"
        )

    return df[["text", "label"]].dropna().reset_index(drop=True)



def load_enron_dataset(
    csv_name: str = ENRON_CSV_NAME,
    dataset_slug: str = ENRON_KAGGLE_DATASET,
    dest_dir: Path = DATA_DIR / "enron",
) -> pd.DataFrame:
    csv_path = _ensure_csv(dataset_slug, csv_name, dest_dir)
    return _load_csv(csv_path, "Enron")


In [3]:
def preprocess_and_vectorize(
    df: pd.DataFrame,
    max_features: int = MAX_TFIDF_FEATURES,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
):
    #TF-IDF features, label encoding, and stratified train/test split.
    texts = df["text"].astype(str)
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df["label"])
    num_classes = len(label_encoder.classes_)

    vectorizer = TfidfVectorizer(max_features=max_features)
    X = vectorizer.fit_transform(texts)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )
    return X_train, X_test, y_train, y_test, num_classes, label_encoder

In [4]:
enron_df = load_enron_dataset()

X_train_e, X_test_e, y_train_e, y_test_e, n_classes_e, le_e = preprocess_and_vectorize(enron_df)


In [5]:
def display_dataset_samples(df: pd.DataFrame, n_samples: int = 5):

    # Random samples
    print(f"--- Displaying {n_samples} Random Examples from the Dataset ---\n")
    
    # Sample random rows for the demonstration
    sample_df = df.sample(n=n_samples, random_state=42)
    
    for index, row in sample_df.iterrows():
        label = row['label']
        # Replace newlines with spaces and truncate to 200 characters for a clean display
        text_preview = str(row['text']).replace('\n', ' ').strip()
        if len(text_preview) > 200:
            text_preview = text_preview[:200] + "..."
            
        print(f"LABEL: [{label.upper()}]")
        print(f"TEXT : {text_preview}")
        print("-" * 80)

# Execute the function
display_dataset_samples(enron_df, n_samples=10)

--- Displaying 10 Random Examples from the Dataset ---

LABEL: [SPAM]
TEXT : re : tenaska iv i tried calling you this am but your phone rolled to someone else ' s voicemail . can you call me when you get a chance ? - - - - - original message - - - - - from : farmer , daren j ....
--------------------------------------------------------------------------------
LABEL: [HAM]
TEXT : neon - bammel neon groups - fall 2001 . doc
--------------------------------------------------------------------------------
LABEL: [SPAM]
TEXT : fw : re ivanhoe e . s . d fyi , kim . - - - - - original message - - - - - from : frazier , perry sent : thursday , march 07 , 2002 2 : 25 pm to : lebeau , randy ; watson , kimberly ; abdmoulaie , man...
--------------------------------------------------------------------------------
LABEL: [SPAM]
TEXT : start date : 2 / 6 / 02 ; hourahead hour : 24 ; start date : 2 / 6 / 02 ; hourahead hour : 24 ; no ancillary schedules awarded . no variances detected . log messages 

In [7]:
def apply_label_flipping(
    y_train: np.ndarray,
    budget: float,
    num_classes: int,
    random_state: int = RANDOM_STATE, 
) -> np.ndarray:
    y_poisoned = np.array(y_train, copy=True)
    n = len(y_poisoned)
    if budget <= 0 or n == 0:
        return y_poisoned

    n_poison = int(np.floor(budget * n))
    if n_poison < 1:
        n_poison = 1
    n_poison = min(n_poison, n)

    rng = np.random.default_rng(random_state) 
    poison_indices = rng.choice(n, size=n_poison, replace=False)

    for idx in poison_indices: # Random flipping either to opposite (binary) or arbitrary class
        original = y_poisoned[idx]
        if num_classes == 2:
            y_poisoned[idx] = 1 - original
        else:
            candidates = [c for c in range(num_classes) if c != original]
            y_poisoned[idx] = rng.choice(candidates)

    return y_poisoned

In [8]:
def apply_targeted_poison(
    y_train: np.ndarray,
    budget: float,
    source_class: int,
    target_class: int,
    random_state: int = RANDOM_STATE,
) -> np.ndarray:
    y_poisoned = np.array(y_train, copy=True)
    n = len(y_poisoned)
    if budget <= 0 or n == 0:
        return y_poisoned

    n_budget = int(np.floor(budget * n))
    if n_budget < 1:
        n_budget = 1

    source_indices = np.where(y_train == source_class)[0]
    
    n_flip = min(n_budget, len(source_indices) - 1)

    if n_flip == 0:
        return y_poisoned

    # safety check: ensure at least 2 classes remain after flipping 
    n_source_remaining = len(source_indices) - n_flip
    if n_source_remaining == 0:
        return y_poisoned  

    rng = np.random.default_rng(random_state)
    flip_indices = rng.choice(source_indices, size=n_flip, replace=False)
    y_poisoned[flip_indices] = target_class

    return y_poisoned

## Budget Analysis

In [ ]:
from ipywidgets import interact, FloatSlider
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

# Pre-compute PCA once outside the widget
pca = PCA(n_components=2, random_state=RANDOM_STATE)
if hasattr(X_train_e, "toarray"):
    X_train_2d = pca.fit_transform(X_train_e.toarray())
    X_test_2d  = pca.transform(X_test_e.toarray())
else:
    X_train_2d = pca.fit_transform(X_train_e)
    X_test_2d  = pca.transform(X_test_e)

# Subsample for speed
rng = np.random.default_rng(RANDOM_STATE)
n_show = 800
idx_tr = rng.choice(len(y_train_e), n_show, replace=False)
idx_te = rng.choice(len(y_test_e),  n_show, replace=False)

# Mesh grid pre-computed once
x_min, x_max = X_train_2d[:, 0].min()-1, X_train_2d[:, 0].max()+1
y_min, y_max = X_train_2d[:, 1].min()-1, X_train_2d[:, 1].max()+1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 150),
    np.linspace(y_min, y_max, 150),
)

def demo_poisoning(budget=0.0, attack="targeted: spam->ham"):
    # Apply poisoning
    if attack == "targeted: spam->ham":
        src = le_e.transform(["spam"])[0]
        tgt = le_e.transform(["ham"])[0]
        y_p = apply_targeted_poison(y_train_e, budget, src, tgt)
    else:
        y_p = apply_label_flipping(y_train_e, budget, n_classes_e)

    n_flipped = int((y_p != y_train_e).sum())

    # Train on poisoned 2D data
    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE) #Logistic Regression
    clf.fit(X_train_2d, y_p)

    # Metrics on clean test set
    y_pred = clf.predict(X_test_2d)
    rec  = recall_score(y_test_e, y_pred, zero_division=0)
    prec = precision_score(y_test_e, y_pred, zero_division=0)
    f1   = f1_score(y_test_e, y_pred, zero_division=0)

    # Decision boundary
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f"Budget: {int(budget*100)}%  |  Labels flipped: {n_flipped:,}  |  "
        f"Precision: {prec:.3f}  Recall: {rec:.3f}  F1: {f1:.3f}",
        fontsize=12, fontweight="bold",
    )

    colors = ["#2196F3", "#F44336"]  # blue=ham, red=spam
    bg_colors = ["#BBDEFB", "#FFCDD2"]

    for ax, (pts, lbls, title) in zip(axes, [
        (X_train_2d[idx_tr], y_p[idx_tr],        "Training Set (poisoned labels)"),
        (X_test_2d[idx_te],  y_test_e[idx_te],   "Test Set (clean labels)"),
    ]):
        from matplotlib.colors import ListedColormap
        ax.contourf(xx, yy, Z, alpha=0.3,
                    cmap=ListedColormap(bg_colors))
        ax.contour(xx, yy, Z, colors="black",
                   linewidths=1.0, alpha=0.6)

        for cls_idx, (cls_name, color) in enumerate(
            zip(le_e.classes_, colors)
        ):
            mask = lbls == cls_idx
            ax.scatter(
                pts[mask, 0], pts[mask, 1],
                color=color, s=15, alpha=0.6,
                edgecolors="none", label=cls_name,
            )

        ax.set_title(title, fontsize=10)
        ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
        ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
        ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

# ── Interactive widget ────
interact(
    demo_poisoning,
    budget=FloatSlider(
        min=0.0, max=0.70, step=0.05,
        value=0.0, description="Budget",
        continuous_update=False,
        style={"description_width": "initial"},
    ),
    attack=["targeted: spam->ham", "untargeted: random"],
)

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='Budget', max=0.7, step=0.05…

<function __main__.demo_poisoning(budget=0.0, attack='targeted: spam->ham')>